# NER 第 2 步：探索数据与可视化（EDA）

版本：2026-11-01 eda-v1。**准备完成，尚待你运行；不需要 Drive 或 bucket。**

按 **2.1 → 2.6 → 下载 ZIP → 2.7 确认** 执行，建议连接第 1 步的运行时。
本笔记本会重新初始化所需变量。若原数据目录不存在，会自动下载同一固定版本，
并与已验收的第 1 步文件哈希及句序哈希核对；不会选择新的数据版本。
本步没有训练或模型评分。测试集统计仅用于文档要求的描述性分析，不用于选择特征或超参数。

交付：13 张 CSV 表、完整 JSON 记录、统计口径说明，以及 **9 组 PNG + PDF 图表**。
每个 run_id 使用独立结果目录；图表全部显示在 2.5，英文图题便于直接用于英文报告。
同时请从「文件 → 下载 .ipynb」保留当前 notebook。

覆盖：论文计数对照、相邻句序抽查、含 O 的标签分布、句长与实体长度、各类型高频实体、
句内标签转移、大小写、未见实体、多类型歧义，以及划分之间的词表重叠。

论文参考：[CoNLL-2003 Tables 1–2](https://aclanthology.org/W03-0419.pdf)。
HF 句子数与论文原始句子数有文档数大小的差异：本步会分别列出实际值、参考值和差值，
再检验“实际句子数 + 文档数”；不会把原始差异隐藏或直接当作错误。


## 2.1 检查依赖
沿用第 0 步版本。已有依赖时不会安装；如提示已安装，请重启 Python 会话，再从 2.1 开始。
本步是 CPU 数据统计，不需要训练 GPU 模型。


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

REQUIRED = {'datasets': '4.0.0', 'numpy': '2.0.2', 'pandas': '2.2.2', 'matplotlib': '3.10.0'}
missing = []
for package, version in REQUIRED.items():
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        missing.append(f'{package}=={version}')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *[f'{p}=={v}' for p,v in REQUIRED.items()]])
    raise RuntimeError('Installation finished. Restart the Python session, then rerun 2.1.')
import datasets
import numpy as np
import pandas as pd
import matplotlib
for package, module in [('datasets', datasets), ('numpy', np), ('pandas', pd), ('matplotlib', matplotlib)]:
    if module.__version__ != REQUIRED[package]:
        raise RuntimeError('A stale package version is imported. Restart the Python session.')
print('Dependencies ready:', REQUIRED)


## 2.2 初始化目录与本步工具
默认目录与前两步一致。若你将项目恢复到其他目录，只修改 `PROJECT_ROOT`。
重跑完整步骤时，从本节开始生成新的 run_id；不要删除以前的备份。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import platform
import random
import uuid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
NOTEBOOK_REVISION = '2026-11-01-eda-v1'
PROJECT_ROOT = Path.home() / 'cs5489_ner_local'
EXPORT_DIR = Path.home() / 'ner_exports'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RESULTS_DIR = PROJECT_ROOT / 'results' / f'step2_{RUN_ID}'
FIGURES_DIR = PROJECT_ROOT / 'figures' / f'step2_{RUN_ID}'
STEP2_COMPLETE = DATA_READY = EDA_VERIFIED = FIGURES_READY = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
for folder in ['config', 'logs', 'src', 'results', 'figures', 'data']:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Run:', RUN_ID)


In [ ]:
SOURCE_BUNDLE = {'local_backup.py': '"""Verified, portable NER backups. Uses local files and the Python standard library."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nFORMAT = \'cs5489-ner-local-backup-v1\'\nALLOWED_FOLDERS = {\'config\', \'logs\', \'src\', \'results\', \'figures\', \'predictions\', \'notebooks\'}\nALLOWED_ROOT_FILES = {\'README.md\', \'requirements.txt\', \'requirements-step0.txt\'}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _relative_path(value):\n    if not isinstance(value, str) or not value or \'\\\\\' in value or \':\' in value:\n        raise ValueError(\'Invalid backup member path.\')\n    parts = value.split(\'/\')\n    if any(part in (\'\', \'.\', \'..\') or part.startswith(\'.\') for part in parts):\n        raise ValueError(\'Unsafe backup member path.\')\n    if PurePosixPath(value).is_absolute():\n        raise ValueError(\'Absolute backup member paths are forbidden.\')\n    return PurePosixPath(value)\n\n\ndef _is_project_output(relative, include_checkpoints):\n    parts = relative.parts\n    if any(part.startswith(\'.\') or part == \'__pycache__\' for part in parts):\n        return False\n    filename = relative.name.lower()\n    if (\'credential\' in filename or \'oauth\' in filename or filename.startswith(\'client_secret\') or\n            filename in {\'token.json\', \'tokens.json\'} or\n            any(secret in filename for secret in (\'access_token\', \'refresh_token\', \'id_token\')) or\n            filename.endswith((\'.zip\', \'.pyc\', \'.pem\', \'.key\'))):\n        return False\n    if len(parts) == 1:\n        return relative.name in ALLOWED_ROOT_FILES\n    return parts[0] in ALLOWED_FOLDERS or (include_checkpoints and parts[0] == \'checkpoints\')\n\n\ndef verify_backup(archive_file, max_total_bytes=20 * 1024**3):\n    """Check structure, file sizes and hashes before accepting a local ZIP.\n\n    Hashes detect corruption; they do not establish who created an archive.\n    Restore only backups from a source you trust.\n    """\n    with zipfile.ZipFile(archive_file) as archive:\n        members = archive.infolist()\n        names = [item.filename for item in members]\n        if len(names) != len(set(names)) or \'manifest.json\' not in names:\n            raise ValueError(\'Backup has duplicate members or no manifest.\')\n        if sum(item.file_size for item in members) > max_total_bytes:\n            raise ValueError(\'Backup exceeds the configured uncompressed-size limit.\')\n        for item in members:\n            _relative_path(item.filename)\n            if stat.S_ISLNK(item.external_attr >> 16) or item.is_dir():\n                raise ValueError(\'Links and directory entries are not valid backup members.\')\n        if archive.getinfo(\'manifest.json\').file_size > 10 * 1024**2:\n            raise ValueError(\'Backup manifest is too large.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if manifest.get(\'format\') != FORMAT or not isinstance(manifest.get(\'files\'), list):\n            raise ValueError(\'Unknown backup format.\')\n        expected_names = {\'manifest.json\'}\n        paths = set()\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            if relative.as_posix() in paths:\n                raise ValueError(\'Duplicate path in backup manifest.\')\n            paths.add(relative.as_posix())\n            member_name = \'project/\' + relative.as_posix()\n            expected_names.add(member_name)\n            if member_name not in names:\n                raise ValueError(\'A manifest file is missing from the ZIP.\')\n            info = archive.getinfo(member_name)\n            if info.file_size != record[\'size_bytes\']:\n                raise ValueError(f\'Backup size check failed: {relative}\')\n            digest = hashlib.sha256()\n            with archive.open(info) as stream:\n                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n                    digest.update(chunk)\n            if digest.hexdigest() != record[\'sha256\']:\n                raise ValueError(f\'Backup checksum check failed: {relative}\')\n        if set(names) != expected_names:\n            raise ValueError(\'Backup contains files not listed in the manifest.\')\n        # Reading every member to EOF above also checks each ZIP CRC.\n    return manifest\n\n\ndef create_backup(project_root, export_dir, step, revision, include_checkpoints=False):\n    """Archive selected project outputs and verify the completed ZIP by reading it."""\n    root = Path(project_root).resolve(strict=True)\n    export = Path(export_dir).resolve()\n    if not root.is_dir() or export.is_relative_to(root):\n        raise ValueError(\'Use an existing project directory and an export directory outside it.\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]+\', step):\n        raise ValueError(\'Use a simple step name, e.g. step0.\')\n    export.mkdir(parents=True, exist_ok=True)\n    records, selected = [], []\n    folders = ALLOWED_FOLDERS | ({\'checkpoints\'} if include_checkpoints else set())\n    candidates = [root / name for name in ALLOWED_ROOT_FILES if (root / name).is_file()]\n    for name in sorted(folders):\n        folder = root / name\n        if folder.is_symlink():\n            raise ValueError(f\'Refusing to back up a linked project directory: {name}\')\n        if folder.is_dir():\n            candidates.extend(folder.rglob(\'*\'))\n    for path in sorted(candidates):\n        relative = path.relative_to(root)\n        if not _is_project_output(relative, include_checkpoints):\n            continue\n        if path.is_symlink() or any((root / Path(*relative.parts[:i])).is_symlink()\n                                    for i in range(1, len(relative.parts))):\n            raise ValueError(f\'Refusing to back up a link: {relative}\')\n        if not path.is_file():\n            continue\n        path.resolve(strict=True).relative_to(root)\n        records.append({\'path\': relative.as_posix(), \'size_bytes\': path.stat().st_size,\n                        \'sha256\': sha256_file(path)})\n        selected.append((path, relative))\n    if not records:\n        raise ValueError(\'No project outputs were selected for backup.\')\n    created = datetime.now(timezone.utc)\n    manifest = {\'format\': FORMAT, \'step\': step, \'revision\': revision,\n                \'created_at_utc\': created.isoformat(), \'files\': records,\n                \'includes_checkpoints\': include_checkpoints}\n    name = f\'NER_{step}_{created:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}.zip\'\n    output = export / name\n    partial = output.with_suffix(\'.partial\')\n    with zipfile.ZipFile(partial, \'x\', compression=zipfile.ZIP_DEFLATED,\n                         compresslevel=2, strict_timestamps=False) as archive:\n        archive.writestr(\'manifest.json\', json.dumps(manifest, indent=2))\n        for path, relative in selected:\n            archive.write(path, arcname=\'project/\' + relative.as_posix())\n    verify_backup(partial)\n    if output.exists():\n        raise FileExistsError(output)\n    partial.rename(output)\n    return {\'path\': str(output), \'name\': output.name, \'size_bytes\': output.stat().st_size,\n            \'sha256\': sha256_file(output), \'file_count\': len(records),\n            \'archive_verified\': True, \'download_confirmed\': False}\n\n\ndef restore_backup(archive_file, destination):\n    """Restore verified bytes into a new directory without executing any code."""\n    manifest = verify_backup(archive_file)\n    target = Path(destination).resolve()\n    if target.exists():\n        raise FileExistsError(\'Choose a new, empty destination name for restoration.\')\n    target.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(archive_file) as archive:\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            path = target.joinpath(*relative.parts)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(\'project/\' + relative.as_posix()) as source, path.open(\'xb\') as output:\n                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)\n            if sha256_file(path) != record[\'sha256\']:\n                raise ValueError(f\'Restored file checksum failed: {relative}\')\n    return {\'project_root\': str(target), \'step\': manifest[\'step\'], \'file_count\': len(manifest[\'files\'])}\n', 'step1_data.py': '"""Pinned CoNLL-2003 loading and validation. Run real data work in Colab only."""\nimport hashlib\nimport json\nfrom datetime import datetime, timezone\nfrom numbers import Integral\nfrom pathlib import Path\n\nREPO_ID = \'tomaarsen/conll2003\'\nREVISION = \'3a0c0677d3177b6122fceed3239dda7a5de6a514\'\nEXPECTED_ROWS = {\'train\': 14041, \'validation\': 3250, \'test\': 3453}\nTAG_FIELDS = (\'pos_tags\', \'chunk_tags\', \'ner_tags\')\nREQUIRED_FIELDS = (\'id\', \'tokens\', *TAG_FIELDS)\nEXPECTED_NER_LABELS = {\'O\'} | {f\'{p}-{t}\' for p in (\'B\', \'I\') for t in (\'PER\', \'ORG\', \'LOC\', \'MISC\')}\nPARQUET_FILES = {s: f\'data/{s}-00000-of-00001.parquet\' for s in EXPECTED_ROWS}\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.partial\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef load_pinned_dataset(project_root):\n    """Download public data, without auth or remote loading scripts; retain file hashes."""\n    import datasets\n    from huggingface_hub import snapshot_download\n\n    if datasets.__version__ != \'4.0.0\':\n        raise RuntimeError(\'Use datasets==4.0.0; install it and restart the Python session.\')\n    cache = Path(project_root) / \'data\' / \'hf_cache\'\n    snapshot = Path(snapshot_download(\n        repo_id=REPO_ID, repo_type=\'dataset\', revision=REVISION,\n        allow_patterns=[*PARQUET_FILES.values(), \'README.md\'],\n        cache_dir=str(cache / \'hub\'), token=False,\n    ))\n    files = {split: str(snapshot / name) for split, name in PARQUET_FILES.items()}\n    source_files = []\n    for name in [*PARQUET_FILES.values(), \'README.md\']:\n        path = snapshot / name\n        if not path.is_file():\n            raise FileNotFoundError(f\'Pinned source file missing: {name}\')\n        source_files.append({\'path\': name, \'size_bytes\': path.stat().st_size, \'sha256\': file_sha256(path)})\n    dataset = datasets.load_dataset(\'parquet\', data_files=files,\n                                    cache_dir=str(cache / \'arrow\'), token=False)\n    provenance = {\n        \'requested_source\': \'eriktks/conll2003\',\n        \'actual_source\': REPO_ID, \'revision\': REVISION, \'format\': \'parquet\',\n        \'source_url\': f\'https://huggingface.co/datasets/{REPO_ID}/tree/{REVISION}\',\n        \'reason\': \'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.\',\n        \'license_as_stated_by_card\': \'other\',\n        \'source_files\': source_files,\n        \'preprocessing\': \'None: no shuffle, split, filtering, label conversion or token normalization.\',\n        \'parity_scope\': \'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. \'\n                        \'Byte-for-byte equivalence to eriktks/conll2003 has not been established.\',\n        \'recorded_at_utc\': datetime.now(timezone.utc).isoformat(),\n    }\n    return dataset, provenance\n\n\ndef label_names(split):\n    names = {}\n    for field in TAG_FIELDS:\n        feature = split.features[field]\n        values = getattr(getattr(feature, \'feature\', None), \'names\', None)\n        if not values or len(values) != len(set(values)):\n            raise ValueError(f\'{field} must have unique ClassLabel names in feature metadata.\')\n        names[field] = list(values)\n    if set(names[\'ner_tags\']) != EXPECTED_NER_LABELS:\n        raise ValueError(\'NER metadata must contain O and B/I labels for PER, ORG, LOC, MISC.\')\n    return names\n\n\ndef audit_dataset(dataset, expected_rows=None):\n    """Reject corrupt/IOB1 data instead of silently repairing labels or dropping rows.\n\n    expected_rows is injectable for tiny offline fixtures; real runs use EXPECTED_ROWS.\n    Hash input: UTF-8 compact JSON for each core record, decoded tag strings, plus LF.\n    """\n    expected_rows = EXPECTED_ROWS if expected_rows is None else expected_rows\n    if set(dataset) != set(expected_rows):\n        raise ValueError(\'Unexpected dataset splits.\')\n    report = {\'splits\': {}, \'label_names\': None, \'iob2_violations\': 0,\n              \'hash_format\': \'core-fields-decoded-tags-jsonl-utf8-v1\'}\n    for split_name, expected in expected_rows.items():\n        split = dataset[split_name]\n        if len(split) != expected:\n            raise ValueError(f\'{split_name}: expected {expected} rows, received {len(split)}.\')\n        if not set(REQUIRED_FIELDS).issubset(split.column_names):\n            raise ValueError(f\'{split_name}: missing required columns.\')\n        names = label_names(split)\n        if report[\'label_names\'] is None:\n            report[\'label_names\'] = names\n        elif names != report[\'label_names\']:\n            raise ValueError(f\'{split_name}: label metadata differs between splits.\')\n        has_documents = {\'document_id\', \'sentence_id\'}.issubset(split.column_names)\n        digest = hashlib.sha256()\n        tokens_total = 0\n        previous_doc, previous_sentence = None, None\n        document_count = 0\n        for index, row in enumerate(split):\n            where = f\'{split_name}[{index}]\'\n            if row[\'id\'] != str(index):\n                raise ValueError(f\'{where}: id/order changed (expected id={index}).\')\n            tokens = row[\'tokens\']\n            if not isinstance(tokens, list) or not tokens or any(not isinstance(t, str) or not t for t in tokens):\n                raise ValueError(f\'{where}: empty or invalid token list.\')\n            if \'-DOCSTART-\' in tokens:\n                raise ValueError(f\'{where}: unexpected DOCSTART sentinel.\')\n            decoded = {}\n            for field in TAG_FIELDS:\n                tags = row[field]\n                if not isinstance(tags, list) or len(tags) != len(tokens):\n                    raise ValueError(f\'{where}: token/{field} length mismatch.\')\n                if any(isinstance(t, bool) or not isinstance(t, Integral) or not 0 <= t < len(names[field]) for t in tags):\n                    raise ValueError(f\'{where}: invalid {field} index.\')\n                decoded[field] = [names[field][int(t)] for t in tags]\n            previous = \'O\'\n            for position, tag in enumerate(decoded[\'ner_tags\']):\n                if tag.startswith(\'I-\') and previous not in (\'B-\' + tag[2:], tag):\n                    raise ValueError(f\'{where} token {position}: illegal IOB2 transition {previous} -> {tag}.\')\n                previous = tag\n            if has_documents:\n                doc, sentence = row[\'document_id\'], row[\'sentence_id\']\n                if any(isinstance(v, bool) or not isinstance(v, Integral) for v in (doc, sentence)):\n                    raise ValueError(f\'{where}: invalid document/sentence id.\')\n                if previous_doc is None or doc != previous_doc:\n                    if sentence != 0 or (previous_doc is not None and doc != previous_doc + 1):\n                        raise ValueError(f\'{where}: document order is not contiguous.\')\n                    document_count += 1\n                elif sentence != previous_sentence + 1:\n                    raise ValueError(f\'{where}: sentence order is not contiguous.\')\n                previous_doc, previous_sentence = doc, sentence\n            record = {\'id\': row[\'id\'], \'tokens\': tokens, **decoded}\n            digest.update((json.dumps(record, ensure_ascii=False, separators=(\',\', \':\')) + \'\\n\').encode(\'utf-8\'))\n            tokens_total += len(tokens)\n        report[\'splits\'][split_name] = {\n            \'rows\': len(split), \'tokens\': tokens_total, \'ordered_core_sha256\': digest.hexdigest(),\n            \'id_order_verified\': True, \'document_order_verified\': has_documents,\n            \'documents\': document_count if has_documents else None,\n        }\n    return report\n\n\ndef save_and_verify(dataset, project_root, run_id, report):\n    """Keep Arrow data on the runtime; verify persistence without putting data in ZIPs."""\n    from datasets import load_from_disk\n\n    destination = Path(project_root) / \'data\' / f\'conll2003_{run_id}\'\n    if destination.exists():\n        raise FileExistsError(\'This run was already saved. Rerun initialization for a new run_id.\')\n    dataset.save_to_disk(str(destination))\n    restored = load_from_disk(str(destination))\n    check = audit_dataset(restored)\n    if check != report:\n        raise ValueError(\'Saved dataset differs after read-back.\')\n    # Includes auxiliary columns and feature metadata, in addition to ordered core hashes.\n    files = [{\'path\': p.relative_to(destination).as_posix(), \'size_bytes\': p.stat().st_size,\n              \'sha256\': file_sha256(p)} for p in sorted(destination.rglob(\'*\')) if p.is_file()]\n    return {\'path\': str(destination), \'read_back_verified\': True, \'files\': files}\n', 'step2_eda.py': '"""Descriptive NER statistics. No training, model selection or test-set scoring."""\nimport csv\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nSPLITS = (\'train\', \'validation\', \'test\')\nTYPES = (\'PER\', \'ORG\', \'LOC\', \'MISC\')\nPAPER = {\n    \'train\': dict(documents=946, paper_sentences=14987, sentences=14041, tokens=203621,\n                  LOC=7140, MISC=3438, ORG=6321, PER=6600),\n    \'validation\': dict(documents=216, paper_sentences=3466, sentences=3250, tokens=51362,\n                       LOC=1837, MISC=922, ORG=1341, PER=1842),\n    \'test\': dict(documents=231, paper_sentences=3684, sentences=3453, tokens=46435,\n                 LOC=1668, MISC=702, ORG=1661, PER=1617),\n}\nDEFINITIONS = {\n    \'entity\': \'Strict IOB2 span, counted once at B-. End index is exclusive.\',\n    \'surface\': \'Exact tuple of original entity tokens; case-sensitive, no punctuation or whitespace normalization.\',\n    \'unseen_primary\': \'Surface absent among ALL training gold entity types. Mention rate weights occurrences; unique rate weights distinct surfaces within each type.\',\n    \'unseen_typed\': \'Surface absent from training gold entities of the SAME type; reported separately.\',\n    \'casefold\': \'Supplemental case-insensitive comparison uses str.casefold() on each token; original data are unchanged.\',\n    \'capitalization\': \'First character of the original first token is uppercase (str.isupper). Entity denominator: all mentions of that type. Sentence-O denominator: sentences whose token at index 0 has O; no punctuation skipping.\',\n    \'transition\': \'Only adjacent tokens within a sentence; no transitions across sentences. Probability divides by outgoing transitions for that label. Zero denominator is null (N/A).\',\n    \'ambiguity\': \'Exact token forms or complete entity surfaces with at least two distinct entity types WITHIN a split. O is excluded from entity types; O occurrence counts are supplemental.\',\n    \'vocabulary\': \'Includes punctuation and digits. Directed source-to-target type coverage = intersection / target vocabulary size; token coverage weights target token occurrences. Jaccard = intersection / union.\',\n    \'top_entities\': \'Top 10 per split and type, by mention count descending then token tuple ascending; plots show training entities.\',\n    \'paper_sentences\': \'Compare HF sentences directly to 14041/3250/3453; also compare HF sentences + document count to paper sentence count. DOCSTART explanation is an inference, not a reconstruction of the original files.\',\n    \'test_use\': \'Test statistics are descriptive only and must not select features, hyperparameters, preprocessing or models. No prediction scores are computed.\',\n}\n\n\ndef ratio(numerator, denominator):\n    return numerator / denominator if denominator else None\n\n\ndef extract_entities(tokens, tags):\n    """Return (start, end, type, token_tuple), rejecting illegal IOB2."""\n    if len(tokens) != len(tags):\n        raise ValueError(\'Token/tag length mismatch.\')\n    spans = []\n    start = kind = None\n    for index, tag in enumerate([*tags, \'O\']):\n        if tag != \'O\' and (tag[:2] not in (\'B-\', \'I-\') or tag[2:] not in TYPES):\n            raise ValueError(f\'Unknown tag: {tag}\')\n        if tag.startswith(\'I-\'):\n            if start is None or kind != tag[2:]:\n                raise ValueError(\'Illegal IOB2 continuation.\')\n        else:\n            if start is not None:\n                spans.append((start, index, kind, tuple(tokens[start:index])))\n                start = kind = None\n            if tag.startswith(\'B-\'):\n                start, kind = index, tag[2:]\n    return spans\n\n\ndef _normal_surface(surface, mode):\n    return surface if mode == \'exact\' else tuple(token.casefold() for token in surface)\n\n\ndef analyze(dataset, labels):\n    """One deterministic traversal; keeps distributions/aggregates rather than full rows."""\n    tables = {name: [] for name in (\'summary\', \'labels\', \'sentence_lengths\', \'entity_lengths\',\n              \'top_entities\', \'transitions\', \'capitalization\', \'unseen\', \'vocabulary\',\n              \'ambiguous_tokens\', \'ambiguous_entities\', \'order_spotchecks\', \'paper_comparison\')}\n    entities_by_split, vocab_by_split = {}, {}\n    for split in SPLITS:\n        rows = dataset[split]\n        label_counts, lengths, transitions, vocab = Counter(), Counter(), Counter(), Counter()\n        entity_counts = {kind: Counter() for kind in TYPES}\n        entity_lengths = {kind: Counter() for kind in TYPES}\n        caps = Counter()\n        token_types, surface_types = defaultdict(Counter), defaultdict(Counter)\n        outside_tokens = Counter()\n        docs, spots = set(), set(range(min(3, len(rows))))\n        middle = len(rows) // 2\n        spots.update(range(max(0, middle - 1), min(len(rows), middle + 2)))\n        previous_doc = None\n        boundary_found = False\n        for index, row in enumerate(rows):\n            tokens = row[\'tokens\']\n            tags = [labels[tag] for tag in row[\'ner_tags\']]\n            spans = extract_entities(tokens, tags)\n            label_counts.update(tags)\n            lengths[len(tokens)] += 1\n            vocab.update(tokens)\n            transitions.update(zip(tags, tags[1:]))\n            docs.add(row[\'document_id\'])\n            if previous_doc is not None and row[\'document_id\'] != previous_doc and not boundary_found:\n                spots.update([index - 1, index])\n                boundary_found = True\n            previous_doc = row[\'document_id\']\n            if tags and tags[0] == \'O\':\n                caps[\'sentence_O_total\'] += 1\n                caps[\'sentence_O_upper\'] += int(tokens[0][:1].isupper())\n            for token, tag in zip(tokens, tags):\n                if tag == \'O\':\n                    outside_tokens[token] += 1\n                else:\n                    token_types[token][tag[2:]] += 1\n            for start, end, kind, surface in spans:\n                entity_counts[kind][surface] += 1\n                entity_lengths[kind][end - start] += 1\n                surface_types[surface][kind] += 1\n                caps[kind] += int(surface[0][:1].isupper())\n            if len(spans) != sum(tag.startswith(\'B-\') for tag in tags):\n                raise ValueError(\'Span/B-tag count mismatch.\')\n        total_tokens = sum(vocab.values())\n        counts = {kind: sum(entity_counts[kind].values()) for kind in TYPES}\n        if sum(label_counts.values()) != total_tokens or sum(transitions.values()) != total_tokens - len(rows):\n            raise ValueError(\'Count/transition conservation failed.\')\n        summary = dict(split=split, sentences=len(rows), tokens=total_tokens, documents=len(docs),\n                       entities=sum(counts.values()), **counts)\n        tables[\'summary\'].append(summary)\n        for metric in (\'sentences\', \'tokens\', \'documents\', *TYPES):\n            expected = PAPER[split][metric]\n            tables[\'paper_comparison\'].append(dict(split=split, metric=metric,\n                observed=summary[metric], reference=expected, difference=summary[metric] - expected))\n        adjusted = len(rows) + len(docs)\n        tables[\'paper_comparison\'].append(dict(split=split, metric=\'sentences_plus_documents\',\n            observed=adjusted, reference=PAPER[split][\'paper_sentences\'],\n            difference=adjusted - PAPER[split][\'paper_sentences\']))\n        tables[\'paper_comparison\'].append(dict(split=split, metric=\'raw_sentences_vs_paper\',\n            observed=len(rows), reference=PAPER[split][\'paper_sentences\'],\n            difference=len(rows) - PAPER[split][\'paper_sentences\'], expected_difference=-PAPER[split][\'documents\']))\n        for label in labels:\n            tables[\'labels\'].append(dict(split=split, label=label, count=label_counts[label],\n                                        proportion=ratio(label_counts[label], total_tokens)))\n            outgoing = sum(transitions[(label, other)] for other in labels)\n            for other in labels:\n                n = transitions[(label, other)]\n                tables[\'transitions\'].append(dict(split=split, previous=label, following=other,\n                                                  count=n, probability=ratio(n, outgoing)))\n        for length, count in sorted(lengths.items()):\n            tables[\'sentence_lengths\'].append(dict(split=split, length=length, count=count))\n        for kind in TYPES:\n            for length, count in sorted(entity_lengths[kind].items()):\n                tables[\'entity_lengths\'].append(dict(split=split, type=kind, length=length,\n                                                     count=count, proportion=ratio(count, counts[kind])))\n            for rank, (surface, count) in enumerate(sorted(entity_counts[kind].items(),\n                                                           key=lambda x: (-x[1], x[0]))[:10], 1):\n                tables[\'top_entities\'].append(dict(split=split, type=kind, rank=rank,\n                                                  surface=\' \'.join(surface), count=count))\n            tables[\'capitalization\'].append(dict(split=split, metric=kind, uppercase=caps[kind],\n                                                 total=counts[kind], proportion=ratio(caps[kind], counts[kind])))\n        tables[\'capitalization\'].append(dict(split=split, metric=\'sentence_initial_O\',\n            uppercase=caps[\'sentence_O_upper\'], total=caps[\'sentence_O_total\'],\n            proportion=ratio(caps[\'sentence_O_upper\'], caps[\'sentence_O_total\'])))\n        for table, forms in [(\'ambiguous_tokens\', token_types), (\'ambiguous_entities\', surface_types)]:\n            ambiguous = [(form, values) for form, values in forms.items() if len(values) > 1]\n            for form, values in sorted(ambiguous, key=lambda x: (-sum(x[1].values()), x[0])):\n                item = dict(split=split, surface=form if table == \'ambiguous_tokens\' else \' \'.join(form),\n                            types=\'|\'.join(sorted(values)), total=sum(values.values()),\n                            **{kind: values[kind] for kind in TYPES})\n                if table == \'ambiguous_tokens\':\n                    item[\'O_count\'] = outside_tokens[form]\n                tables[table].append(item)\n        for index in sorted(spots):\n            row = rows[index]\n            tables[\'order_spotchecks\'].append(dict(split=split, row=index, id=row[\'id\'],\n                document_id=row[\'document_id\'], sentence_id=row[\'sentence_id\'],\n                preview=\' \'.join(row[\'tokens\'][:24])))\n        entities_by_split[split], vocab_by_split[split] = entity_counts, vocab\n\n    for mode in (\'exact\', \'casefold\'):\n        train_by_type = {kind: {_normal_surface(s, mode) for s in entities_by_split[\'train\'][kind]}\n                         for kind in TYPES}\n        train_any = set().union(*train_by_type.values())\n        for split in (\'validation\', \'test\'):\n            for kind in TYPES:\n                target = Counter()\n                for surface, count in entities_by_split[split][kind].items():\n                    target[_normal_surface(surface, mode)] += count\n                unseen = {s for s in target if s not in train_any}\n                unseen_typed = {s for s in target if s not in train_by_type[kind]}\n                n, unique = sum(target.values()), len(target)\n                tables[\'unseen\'].append(dict(split=split, normalization=mode, type=kind,\n                    mentions=n, unique_surfaces=unique,\n                    unseen_mentions=sum(target[s] for s in unseen), unseen_unique=len(unseen),\n                    mention_rate=ratio(sum(target[s] for s in unseen), n), unique_rate=ratio(len(unseen), unique),\n                    unseen_typed_mentions=sum(target[s] for s in unseen_typed), unseen_typed_unique=len(unseen_typed),\n                    typed_mention_rate=ratio(sum(target[s] for s in unseen_typed), n),\n                    typed_unique_rate=ratio(len(unseen_typed), unique)))\n        vocabs = {}\n        for split in SPLITS:\n            values = Counter()\n            for token, count in vocab_by_split[split].items():\n                values[token if mode == \'exact\' else token.casefold()] += count\n            vocabs[split] = values\n        for source in SPLITS:\n            for target in SPLITS:\n                a, b = set(vocabs[source]), set(vocabs[target])\n                overlap = a & b\n                covered = sum(vocabs[target][t] for t in overlap)\n                tables[\'vocabulary\'].append(dict(normalization=mode, source=source, target=target,\n                    source_types=len(a), target_types=len(b), shared_types=len(overlap), union_types=len(a | b),\n                    type_coverage=ratio(len(overlap), len(b)), jaccard=ratio(len(overlap), len(a | b)),\n                    covered_target_tokens=covered, target_tokens=sum(vocabs[target].values()),\n                    token_coverage=ratio(covered, sum(vocabs[target].values()))))\n    for row in tables[\'paper_comparison\']:\n        row.setdefault(\'expected_difference\', 0)\n        row[\'matches_expected\'] = row[\'difference\'] == row[\'expected_difference\']\n    return {\'definitions\': DEFINITIONS, \'labels_from_metadata\': list(labels), \'tables\': tables,\n            \'paper_comparison_passed\': all(row[\'matches_expected\'] for row in tables[\'paper_comparison\'])}\n\n\ndef ensure_dataset(project_root, accepted, run_id):\n    """Reuse the accepted Step 1 files, or redownload exactly the same pinned content."""\n    import step1_data as data_tools\n    from datasets import load_from_disk\n\n    project_root = Path(project_root)\n    path = project_root / \'data\' / Path(accepted[\'saved_data\'][\'path\']).name\n    if path.is_dir():\n        for record in accepted[\'saved_data\'][\'files\']:\n            file = path / record[\'path\']\n            if not file.is_file() or data_tools.file_sha256(file) != record[\'sha256\']:\n                raise ValueError(\'Saved Step 1 data changed. Stop and report this error.\')\n        dataset = load_from_disk(str(path))\n        source = accepted[\'source\']\n        recovery = \'reused-verified-step1-files\'\n    else:\n        dataset, source = data_tools.load_pinned_dataset(project_root)\n        if source[\'source_files\'] != accepted[\'source\'][\'source_files\']:\n            raise ValueError(\'Pinned downloads differ from the accepted Step 1 source hashes.\')\n        recovery = \'redownloaded-pinned-step1-source\'\n    audit = data_tools.audit_dataset(dataset)\n    if audit != accepted[\'audit\']:\n        raise ValueError(\'Dataset differs from the accepted Step 1 ordered audit.\')\n    if recovery.startswith(\'redownloaded\'):\n        saved = data_tools.save_and_verify(dataset, project_root, run_id, audit)\n    else:\n        saved = {**accepted[\'saved_data\'], \'path\': str(path)}\n    record = dict(source=source, saved_data=saved, audit=audit, recovery=recovery, run_id=run_id)\n    data_tools.write_json(project_root / \'config\' / \'dataset_latest.json\', record)\n    return dataset, record\n\n\ndef write_tables(report, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=False)\n    for name, rows in report[\'tables\'].items():\n        path = destination / f\'{name}.csv\'\n        with path.open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as stream:\n            fields = list(rows[0]) if rows else [\'split\', \'surface\', \'types\', \'total\', *TYPES]\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    (destination / \'eda_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    lines = [\'# Step 2 descriptive EDA\', \'\', \'No model training or prediction scoring.\', \'\',\n             \'Reference: https://aclanthology.org/W03-0419.pdf (Tables 1 and 2).\', \'\', \'## Definitions\', \'\']\n    lines.extend(f\'- {key}: {value}\' for key, value in report[\'definitions\'].items())\n    lines += [\'\', \'## Observed counts\', \'\', \'| Split | Sentences | Tokens | Entities | PER | ORG | LOC | MISC |\',\n              \'|---|---:|---:|---:|---:|---:|---:|---:|\']\n    for row in report[\'tables\'][\'summary\']:\n        lines.append(\'| \' + \' | \'.join(str(row[k]) for k in (\'split\', \'sentences\', \'tokens\', \'entities\', *TYPES)) + \' |\')\n    lines += [\'\', f"Paper count comparison passed: {report[\'paper_comparison_passed\']}.",\n              \'Raw paper sentence counts include an additional document-sized difference; see paper_comparison.csv.\',\n              \'Use train-only observations when considering features. Test statistics are descriptive, not a selection criterion.\',\n              \'PNG and vector PDF figures are stored in the corresponding figures/step2_<run_id> directory.\']\n    (destination / \'EDA_README.md\').write_text(\'\\n\'.join(lines) + \'\\n\', encoding=\'utf-8\')\n\n\nFIGURE_NAMES = (\'01_label_distribution\', \'02_sentence_lengths\', \'03_entity_lengths\',\n                \'04_top_entities_train\', \'05_tag_transitions\', \'06_capitalization\',\n                \'07_unseen_entities\', \'08_vocabulary_overlap\', \'09_ambiguity_train\')\n\n\ndef plot_report(report, destination):\n    """Nine report figures, each exported as a PNG and a vector PDF."""\n    import matplotlib.pyplot as plt\n    import numpy as np\n    from matplotlib.ticker import PercentFormatter, MaxNLocator\n    from textwrap import fill\n\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    tables, labels = report[\'tables\'], report[\'labels_from_metadata\']\n    colors = [\'#2670AA\', \'#D17B22\', \'#36846B\', \'#A75085\']\n    paths = []\n    style = {\'font.family\': \'DejaVu Sans\', \'font.size\': 10, \'axes.spines.top\': False,\n             \'axes.spines.right\': False, \'savefig.facecolor\': \'white\', \'figure.facecolor\': \'white\'}\n\n    def selected(name, **where):\n        return [r for r in tables[name] if all(r[k] == v for k, v in where.items())]\n\n    def save(fig, name):\n        fig.savefig(destination / (name + \'.png\'), dpi=170, bbox_inches=\'tight\')\n        fig.savefig(destination / (name + \'.pdf\'), bbox_inches=\'tight\')\n        paths.append(str(destination / (name + \'.png\')))\n        plt.close(fig)\n\n    def number(value):\n        return float(\'nan\') if value is None else value\n\n    with plt.rc_context(style):\n        fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), layout=\'constrained\', sharey=True)\n        for ax, split, color in zip(axes, SPLITS, colors):\n            rows = selected(\'labels\', split=split)\n            ax.bar([r[\'label\'] for r in rows], [r[\'proportion\'] for r in rows], color=color)\n            ax.set(title=split, xlabel=\'Token label (includes O)\')\n            ax.tick_params(axis=\'x\', rotation=55)\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n        axes[0].set_ylabel(\'Share of tokens\')\n        fig.suptitle(\'NER label distribution\')\n        save(fig, FIGURE_NAMES[0])\n\n        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout=\'constrained\')\n        maximum = max(r[\'length\'] for r in tables[\'sentence_lengths\'])\n        bins = np.arange(0.5, maximum + 5.5, 5)\n        for ax, split, color in zip(axes, SPLITS, colors):\n            rows = selected(\'sentence_lengths\', split=split)\n            ax.hist([r[\'length\'] for r in rows], bins=bins, weights=[r[\'count\'] for r in rows], color=color)\n            ax.set(title=f\'{split} (max {max(r["length"] for r in rows)})\', xlabel=\'Tokens per sentence\', ylabel=\'Sentences\', xlim=(0, maximum + 1))\n        fig.suptitle(\'Sentence lengths — all sentences, no tail truncation\')\n        save(fig, FIGURE_NAMES[1])\n\n        fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout=\'constrained\')\n        for ax, kind in zip(axes.flat, TYPES):\n            kind_rows = selected(\'entity_lengths\', type=kind)\n            maximum = max((r[\'length\'] for r in kind_rows), default=1)\n            x = np.arange(1, maximum + 1)\n            for split, color in zip(SPLITS, colors):\n                rows = selected(\'entity_lengths\', split=split, type=kind)\n                counts = {r[\'length\']: r[\'count\'] for r in rows}\n                total = sum(counts.values())\n                probabilities = [counts.get(length, 0) / total if total else np.nan for length in x]\n                ax.plot(x, probabilities, marker=\'o\', ms=3, label=split, color=color)\n            ax.set(title=kind, xlabel=\'Tokens per entity\', ylabel=\'Share within split/type\',\n                   ylim=(0, 1.05), xlim=(.5, maximum + .5))\n            ax.xaxis.set_major_locator(MaxNLocator(integer=True))\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n            ax.legend(fontsize=8)\n        fig.suptitle(\'Entity length distributions\')\n        save(fig, FIGURE_NAMES[2])\n\n        fig, axes = plt.subplots(2, 2, figsize=(16, 12), layout=\'constrained\')\n        for ax, kind, color in zip(axes.flat, TYPES, colors):\n            rows = selected(\'top_entities\', split=\'train\', type=kind)\n            ax.barh(range(len(rows)), [r[\'count\'] for r in rows], color=color)\n            ax.set_yticks(range(len(rows)), [fill(r[\'surface\'], 26) for r in rows], fontsize=9)\n            ax.invert_yaxis()\n            ax.set(title=kind, xlabel=\'Entity mentions in training data\')\n        fig.suptitle(\'Ten most frequent entity surfaces per type (train only)\')\n        save(fig, FIGURE_NAMES[3])\n\n        fig, axes = plt.subplots(1, 3, figsize=(20, 7), layout=\'constrained\')\n        for ax, split in zip(axes, SPLITS):\n            lookup = {(r[\'previous\'], r[\'following\']): r[\'probability\'] for r in selected(\'transitions\', split=split)}\n            matrix = np.array([[number(lookup[(a, b)]) for b in labels] for a in labels])\n            im = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap=\'Blues\')\n            ax.set_xticks(range(len(labels)), labels, rotation=60, ha=\'right\')\n            ax.set_yticks(range(len(labels)), labels)\n            ax.set(title=split, xlabel=\'Next label\', ylabel=\'Current label\')\n            for i in range(len(labels)):\n                for j in range(len(labels)):\n                    v = matrix[i, j]\n                    ax.text(j, i, \'N/A\' if np.isnan(v) else f\'{100*v:.0f}\', ha=\'center\', va=\'center\',\n                            fontsize=7, color=\'white\' if v > .55 else \'#172B40\')\n        fig.colorbar(im, ax=axes, shrink=.65, label=\'Row probability\')\n        fig.suptitle(\'Within-sentence label transitions — cell values in percent\')\n        save(fig, FIGURE_NAMES[4])\n\n        fig, ax = plt.subplots(figsize=(12, 5.5), layout=\'constrained\')\n        categories = [*TYPES, \'sentence_initial_O\']\n        x = np.arange(len(categories))\n        for i, (split, color) in enumerate(zip(SPLITS, colors)):\n            lookup = {r[\'metric\']: r for r in selected(\'capitalization\', split=split)}\n            ax.bar(x + (i-1)*.25, [number(lookup[k][\'proportion\']) for k in categories], width=.24, color=color, label=split)\n        ax.set_xticks(x, [*TYPES, \'Sentence-first O token\'])\n        ax.set(ylabel=\'Uppercase first character / eligible occurrences\', ylim=(0, 1.05),\n               title=\'Capitalization of entity starts and sentence-initial non-entities\')\n        ax.yaxis.set_major_formatter(PercentFormatter(1))\n        ax.legend()\n        save(fig, FIGURE_NAMES[5])\n\n        fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout=\'constrained\', sharey=True)\n        x = np.arange(len(TYPES))\n        for ax, rate, title in zip(axes, (\'mention_rate\', \'unique_rate\'), (\'By mention occurrence\', \'By unique surface\')):\n            for i, (split, color) in enumerate(zip((\'validation\', \'test\'), colors[1:])):\n                lookup = {r[\'type\']: r for r in selected(\'unseen\', split=split, normalization=\'exact\')}\n                ax.bar(x + (i-.5)*.34, [number(lookup[k][rate]) for k in TYPES], width=.33, label=split, color=color)\n            ax.set_xticks(x, TYPES)\n            ax.set(title=title, ylabel=\'Unseen proportion\', ylim=(0, 1.05))\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n            ax.legend()\n        fig.suptitle(\'Entity surfaces unseen among training entities (exact case, any type)\')\n        save(fig, FIGURE_NAMES[6])\n\n        fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), layout=\'constrained\')\n        for ax, mode in zip(axes, (\'exact\', \'casefold\')):\n            lookup = {(r[\'source\'], r[\'target\']): r[\'type_coverage\'] for r in selected(\'vocabulary\', normalization=mode)}\n            matrix = np.array([[number(lookup[(a, b)]) for b in SPLITS] for a in SPLITS])\n            im = ax.imshow(matrix, cmap=\'YlGnBu\', vmin=0, vmax=1)\n            ax.set_xticks(range(3), SPLITS)\n            ax.set_yticks(range(3), SPLITS)\n            ax.set(title=mode, xlabel=\'Target vocabulary (denominator)\', ylabel=\'Source vocabulary\')\n            for i in range(3):\n                for j in range(3):\n                    ax.text(j, i, f\'{matrix[i,j]:.1%}\', ha=\'center\', va=\'center\', color=\'white\' if matrix[i,j] > .65 else \'black\')\n        fig.colorbar(im, ax=axes, shrink=.75, label=\'Target type coverage\')\n        fig.suptitle(\'Directed vocabulary overlap (includes punctuation and digits)\')\n        save(fig, FIGURE_NAMES[7])\n\n        fig, axes = plt.subplots(1, 2, figsize=(16, 10), layout=\'constrained\')\n        for ax, name, title in zip(axes, (\'ambiguous_tokens\', \'ambiguous_entities\'), (\'Token forms\', \'Complete entity surfaces\')):\n            rows = selected(name, split=\'train\')[:15]\n            if not rows:\n                ax.text(.5, .5, \'No multi-type forms\', ha=\'center\', va=\'center\', transform=ax.transAxes)\n                ax.set_title(title)\n                continue\n            left = np.zeros(len(rows))\n            for kind, color in zip(TYPES, colors):\n                values = np.array([r[kind] for r in rows])\n                ax.barh(range(len(rows)), values, left=left, label=kind, color=color)\n                left += values\n            ax.set_yticks(range(len(rows)), [fill(r[\'surface\'], 25) for r in rows], fontsize=9)\n            ax.invert_yaxis()\n            ax.set(title=title, xlabel=\'Entity-token occurrences\' if name == \'ambiguous_tokens\' else \'Entity mentions\')\n            ax.legend(fontsize=8)\n        fig.suptitle(\'Multi-type ambiguity in training data (up to 15 most frequent forms; O excluded)\')\n        save(fig, FIGURE_NAMES[8])\n    return paths\n'}
ACCEPTED_STEP1 = {'run_id': '20261101T071156Z-3bdbd601', 'source': {'requested_source': 'eriktks/conll2003', 'actual_source': 'tomaarsen/conll2003', 'revision': '3a0c0677d3177b6122fceed3239dda7a5de6a514', 'format': 'parquet', 'source_url': 'https://huggingface.co/datasets/tomaarsen/conll2003/tree/3a0c0677d3177b6122fceed3239dda7a5de6a514', 'reason': 'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.', 'license_as_stated_by_card': 'other', 'source_files': [{'path': 'data/train-00000-of-00001.parquet', 'size_bytes': 1241085, 'sha256': '28924b53e43ec4d741cbef16d8a6245a77f404a77f18d44e256f95fa29acb59b'}, {'path': 'data/validation-00000-of-00001.parquet', 'size_bytes': 315597, 'sha256': '0aa0bd59cf9ac8f60252d8194a8a223b650daba50f8e5a79284abf85420bfe84'}, {'path': 'data/test-00000-of-00001.parquet', 'size_bytes': 287552, 'sha256': '9242163e112f99b43c32b663c6d6f7d5df64de73e19e02fcac789935f72565b8'}, {'path': 'README.md', 'size_bytes': 14967, 'sha256': 'a6812e0cca383af9f4c316f8abe43169ddda7230c2686ba314629e59a41a66b9'}], 'preprocessing': 'None: no shuffle, split, filtering, label conversion or token normalization.', 'parity_scope': 'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. Byte-for-byte equivalence to eriktks/conll2003 has not been established.', 'recorded_at_utc': '2026-11-01T07:12:08.589360+00:00'}, 'saved_data': {'path': '/root/cs5489_ner_local/data/conll2003_20261101T071156Z-3bdbd601', 'read_back_verified': True, 'files': [{'path': 'dataset_dict.json', 'size_bytes': 43, 'sha256': '1239dbd4ca38311b03319b672c05a6fdaacc884b0034eed047659406c62eedd4'}, {'path': 'test/data-00000-of-00001.arrow', 'size_bytes': 1614264, 'sha256': 'c640eee74c9972843f89bbebd941fcb18556d81218ded7184f13c21b6bf216f6'}, {'path': 'test/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'test/state.json', 'size_bytes': 249, 'sha256': 'c892cbd70fc09753477fb603f0e74c29336ba93c29edba222e5c98f5a01c452b'}, {'path': 'train/data-00000-of-00001.arrow', 'size_bytes': 7056024, 'sha256': 'd62e5b846961b150ea80d24d2ee411f1eb63365b6aa9d91631057a61246b755f'}, {'path': 'train/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'train/state.json', 'size_bytes': 250, 'sha256': 'f6a54261290f52000b71677342f5a2e37f51e501014abe57b51afc6bbbc35fc0'}, {'path': 'validation/data-00000-of-00001.arrow', 'size_bytes': 1769816, 'sha256': 'ab07b4269a275b8a87b240b9169e245a647ab95353027f417118f321d16d5ee8'}, {'path': 'validation/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'validation/state.json', 'size_bytes': 255, 'sha256': '3525e2995d3da357bb60a26ef20cc66059d0645bda88ad351954da7f7bef5a0a'}]}, 'audit': {'splits': {'train': {'rows': 14041, 'tokens': 203621, 'ordered_core_sha256': 'aa1208d2f06bc11610d0d9527af175e282ca0f4092a06a5a9381240e1dfece6a', 'id_order_verified': True, 'document_order_verified': True, 'documents': 946}, 'validation': {'rows': 3250, 'tokens': 51362, 'ordered_core_sha256': '91c3c12a9140e03ca679f24dc0437cdbae033babea946388677e46f9a52a78c2', 'id_order_verified': True, 'document_order_verified': True, 'documents': 216}, 'test': {'rows': 3453, 'tokens': 46435, 'ordered_core_sha256': 'bbdf60c2c1c3fbd1e7cb6a865d9d4dfd9df0110dff115de49f972175df7d3882', 'id_order_verified': True, 'document_order_verified': True, 'documents': 231}}, 'label_names': {'pos_tags': ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], 'chunk_tags': ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], 'ner_tags': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']}, 'iob2_violations': 0, 'hash_format': 'core-fields-decoded-tags-jsonl-utf8-v1'}, 'acceptance': {'date': '2026-11-01', 'evidence': 'NER_step1_20261101T071221Z_c89d59a3.zip', 'archive_sha256': 'e7094d447aa506d81c3b2c724df668bfdbfb7bc7cb73cebc205178e8728b9841'}}
for name, source in SOURCE_BUNDLE.items():
    (PROJECT_ROOT / 'src' / name).write_text(source, encoding='utf-8')

def load_project_module(name):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / 'src' / (name + '.py'))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

backup = load_project_module('local_backup')
data_tools = load_project_module('step1_data')
eda_tools = load_project_module('step2_eda')
data_tools.write_json(PROJECT_ROOT / 'config' / 'step1_accepted_reference.json', ACCEPTED_STEP1)
print('EDA and backup tools ready; accepted Step 1 reference loaded.')


## 2.3 读取并核对第 1 步数据
优先使用原来的 Arrow 文件，并验证文件哈希；数据缺失时自动重新下载。
所有路径都在运行时本地。必须与第 1 步的来源文件和逐句内容哈希一致才继续。


In [ ]:
STEP2_COMPLETE = DATA_READY = EDA_VERIFIED = FIGURES_READY = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
raw_datasets, DATA_RECORD = eda_tools.ensure_dataset(PROJECT_ROOT, ACCEPTED_STEP1, RUN_ID)
LABEL_NAMES = DATA_RECORD['audit']['label_names']['ner_tags']
DATA_READY = True
print('Data verified against accepted Step 1:', DATA_RECORD['recovery'])
print('Dataset path:', DATA_RECORD['saved_data']['path'])
print('Rows:', {s:r['rows'] for s,r in DATA_RECORD['audit']['splits'].items()})


## 2.4 计算统计并与论文对照
- 实体数以完整 IOB2 span 计数，并独立核对 B- 标签数。
- 大写比例使用首词的第一个字符；句首 O 的分母是“第一个词为 O 的句子”，不跳过标点。
- 未见实体主口径：完整名称未作为任何类型的实体出现在训练集；另保存同类型口径。
  两者都区分出现次数与不同名称数量，并给出大小写敏感 / casefold 两种版本。
- 标签转移不跨句；词表重叠包含标点，保存有方向的覆盖率、Jaccard 和按词频加权的覆盖率。
- 词形歧义与完整实体歧义分别统计，各划分独立，图表只展示训练集歧义。

本节显示计数、论文差异、训练句序抽查、未见实体比例和训练集歧义样例。
若论文可比计数不一致，会保留差异表并停止验收；不要手动改数字使其通过。


In [ ]:
from IPython.display import display, Image

STEP2_COMPLETE = EDA_VERIFIED = FIGURES_READY = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not DATA_READY:
    raise RuntimeError('Complete 2.3 first.')
EDA_REPORT = eda_tools.analyze(raw_datasets, LABEL_NAMES)
EDA_REPORT['provenance'] = {
    'run_id': RUN_ID, 'notebook_revision': NOTEBOOK_REVISION,
    'source': DATA_RECORD['source'], 'step1_ordered_audit': DATA_RECORD['audit'],
}
eda_tools.write_tables(EDA_REPORT, RESULTS_DIR)
print('Observed counts:')
display(pd.DataFrame(EDA_REPORT['tables']['summary']))
print('Paper/HF count comparison (adjusted sentence comparison is shown separately):')
display(pd.DataFrame(EDA_REPORT['tables']['paper_comparison']))
print('Adjacent training sentences (first, middle, first document boundary):')
display(pd.DataFrame([r for r in EDA_REPORT['tables']['order_spotchecks'] if r['split']=='train']))
print('Unseen entity rates (exact case; surfaces absent from all training entity types):')
display(pd.DataFrame([r for r in EDA_REPORT['tables']['unseen'] if r['normalization']=='exact'])[
    ['split','type','unseen_mentions','mentions','mention_rate','unseen_unique','unique_surfaces','unique_rate']])
print('Training token-form ambiguity (up to 15 forms):')
display(pd.DataFrame([r for r in EDA_REPORT['tables']['ambiguous_tokens'] if r['split']=='train'][:15]))
if not EDA_REPORT['paper_comparison_passed']:
    raise RuntimeError('Paper count differences found. Send the comparison table before continuing.')
EDA_VERIFIED = True
print('EDA COUNTS VERIFIED. Tables saved:', RESULTS_DIR)


## 2.5 生成并检查 9 组图表
每组导出 PNG（170 dpi）和矢量 PDF。句长与实体长度保留完整范围。
下面会依次显示所有图表；PNG、PDF 和对应数值表都会进入备份 ZIP。


In [ ]:
STEP2_COMPLETE = FIGURES_READY = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not EDA_VERIFIED:
    raise RuntimeError('Complete 2.4 first.')
FIGURE_PATHS = eda_tools.plot_report(EDA_REPORT, FIGURES_DIR)
if len(FIGURE_PATHS) != 9:
    raise RuntimeError('Expected 9 figure groups.')
for path in FIGURE_PATHS:
    png, pdf = Path(path), Path(path).with_suffix('.pdf')
    if not png.is_file() or png.stat().st_size == 0 or not pdf.is_file() or pdf.stat().st_size == 0:
        raise RuntimeError('A figure export is missing or empty.')
    print(png.name)
    display(Image(filename=str(png)))
FIGURES_READY = True
print('FIGURES READY: 9 PNG + 9 PDF files.')


## 2.6 保存运行记录、生成并下载备份
ZIP 包含本步 CSV/JSON、PNG/PDF、工具源码和来源记录；不包含数据集缓存、模型或凭证。
记录每个结果文件的大小和 SHA-256，并在生成后读回校验 ZIP。


In [ ]:
STEP2_COMPLETE = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not (DATA_READY and EDA_VERIFIED and FIGURES_READY):
    raise RuntimeError('Complete data, statistics and figure checks first.')
ARTIFACTS = []
for directory in (RESULTS_DIR, FIGURES_DIR):
    for path in sorted(directory.iterdir()):
        if path.is_file():
            ARTIFACTS.append({'path':path.relative_to(PROJECT_ROOT).as_posix(),
                              'size_bytes':path.stat().st_size, 'sha256':backup.sha256_file(path)})
RUNTIME_RECORD = {
    'run_id':RUN_ID, 'notebook_revision':NOTEBOOK_REVISION, 'seed':SEED,
    'python':sys.version, 'platform':platform.platform(),
    'packages':{name:metadata.version(name) for name in [*REQUIRED, 'huggingface-hub', 'pyarrow']},
    'data_path':DATA_RECORD['saved_data']['path'], 'data_recovery':DATA_RECORD['recovery'],
    'paper_comparison_passed':EDA_REPORT['paper_comparison_passed'],
    'figure_groups':len(FIGURE_PATHS), 'artifacts':ARTIFACTS,
}
data_tools.write_json(PROJECT_ROOT / 'logs' / f'step2_environment_{RUN_ID}.json', RUNTIME_RECORD)
data_tools.write_json(PROJECT_ROOT / 'config' / 'eda_latest.json', {
    'run_id':RUN_ID, 'results_dir':str(RESULTS_DIR), 'figures_dir':str(FIGURES_DIR),
    'artifacts':ARTIFACTS, 'statistics_verified':True, 'computer_download_confirmed':False,
})
BACKUP_RECEIPT = backup.create_backup(PROJECT_ROOT, EXPORT_DIR, step='step2',
                                    revision=NOTEBOOK_REVISION, include_checkpoints=False)
BACKUP_PATH = Path(BACKUP_RECEIPT['path'])
BACKUP_VERIFIED = BACKUP_RECEIPT['archive_verified']
print('ZIP READY — check the computer copy after downloading.')
print(json.dumps(BACKUP_RECEIPT, indent=2))


In [ ]:
if not BACKUP_VERIFIED or not BACKUP_PATH.is_file():
    raise RuntimeError('Generate the verified ZIP first.')
try:
    from google.colab import files
    files.download(str(BACKUP_PATH))
    print('Download requested:', BACKUP_PATH.name)
except Exception as exc:
    print('Automatic download did not finish:', type(exc).__name__, str(exc))
print('Files-panel fallback path:', BACKUP_PATH)
print('Also save this notebook with File > Download .ipynb.')


## 2.7 确认电脑备份并反馈
确认电脑上有同名且字节数一致的 ZIP 后，将 `BACKUP_DOWNLOADED` 改成 `True`。
浏览过 2.5 的图表后，将 `FIGURES_REVIEWED` 改成 `True`。
发送最后输出；也可将下载的 ZIP 放进电脑的 `CS 5489/NER` 文件夹，方便我复核结果。
如出现报错或图表缺字/被截断，反馈小节与错误文本或相应图表。


In [ ]:
BACKUP_DOWNLOADED = False  # Set True only after checking the computer ZIP.
FIGURES_REVIEWED = False   # Set True after viewing the figures in 2.5.
STEP2_COMPLETE = False
if not (DATA_READY and EDA_VERIFIED and FIGURES_READY and BACKUP_VERIFIED):
    raise RuntimeError('Complete 2.3–2.6 first.')
if not (BACKUP_DOWNLOADED and FIGURES_REVIEWED):
    print('WAITING: confirm the computer ZIP and review the 9 figures, then set both flags True.')
else:
    backup.verify_backup(BACKUP_PATH)
    if backup.sha256_file(BACKUP_PATH) != BACKUP_RECEIPT['sha256']:
        raise RuntimeError('The ZIP changed after its receipt was generated.')
    STEP2_COMPLETE = True
    print('STEP 2 COMPLETE: EDA and ZIP verified; computer download and figure review confirmed by the user.')
    print('Paper count comparison: PASS')
    print('Entities:', {r['split']: {t:r[t] for t in eda_tools.TYPES} for r in EDA_REPORT['tables']['summary']})
    print('Test unseen entity mention rates (exact surface, any training type):',
          {r['type']:round(r['mention_rate'],4) for r in EDA_REPORT['tables']['unseen']
           if r['split']=='test' and r['normalization']=='exact'})
    print('Figures: 9 PNG + 9 PDF')
    print('Tables:', len(EDA_REPORT['tables']))
    print('Results:', RESULTS_DIR)
    print('Backup:', BACKUP_RECEIPT['name'])
    print('Bytes:', BACKUP_RECEIPT['size_bytes'])
    print('SHA-256:', BACKUP_RECEIPT['sha256'])
    print('Send this output before Step 3.')
